# Récupération et segmentation d'illustrations — BSB / MDZ Munich

Télécharge des pages numérisées depuis la bibliothèque numérique de Munich (BSB / MDZ) via IIIF, puis segmente automatiquement les illustrations avec un modèle YOLO.

Le notebook est organisé en deux parties indépendantes :

- **Partie A** — pipeline générique : téléchargement et segmentation d'un ou plusieurs ouvrages BSB dont on connaît déjà l'identifiant.
- **Partie B** — constitution d'un corpus thématique : recherche de documents sur le portail MDZ (ex. Bibles illustrées 1551-1750), sélection manuelle via un tableau HTML interactif, puis extraction en masse des illustrations du corpus retenu.

**Sources** :
- Manifest IIIF : `https://api.digitale-sammlungen.de/iiif/presentation/v2/{id}/manifest`
- API de recherche MDZ : `https://www.digitale-sammlungen.de/api/search`

# Partie A — Téléchargement et segmentation d'un ouvrage BSB (pipeline générique)

## 1. Imports et configuration

In [ ]:
import os, sys

# Chemins — adapter selon l'emplacement du notebook
RACINE = os.path.abspath("../..")
sys.path.insert(0, os.path.join(RACINE, "notebooks"))
sys.path.insert(0, RACINE)
sys.path.insert(0, os.path.join(RACINE, "yolov5_repo"))

from gallica_utils import telecharger_pages_iiif, segmenter_corpus, charger_yolo, liberer_yolo

DOSSIER_SOURCES = os.path.join(RACINE, "data", "sources")
DOSSIER_SEG     = os.path.join(RACINE, "data", "segmentees")

print(f"Racine     : {RACINE}")
print(f"Sources    : {DOSSIER_SOURCES}")
print(f"Segmentées : {DOSSIER_SEG}")

## 2. Sources à récupérer

Renseigner l'identifiant BSB et le nom du dossier de destination (convention `{technique}_{graveur}_{editeur}_{ville}{annee}`).

In [ ]:
SOURCES_BSB = {
    # "bsb_id" : "nom_dossier",
    "bsb00008186" : "cuivre_exemple_editeur_ville1610",
}

## 3. Vérification — nombre de pages disponibles

In [ ]:
import requests

for bsb_id, nom in SOURCES_BSB.items():
    url = f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest"
    r   = requests.get(url, timeout=15)
    if r.status_code == 200:
        pages = len(r.json()["sequences"][0]["canvases"])
        print(f"OK  {nom:45s} : {pages} pages")
    else:
        print(f"ERR {nom:45s} : HTTP {r.status_code}")

## 4. Charger le modèle YOLO

In [ ]:
modele_yolo = charger_yolo()
print("YOLO chargé")

## 5. Téléchargement et segmentation

In [ ]:
for bsb_id, nom in SOURCES_BSB.items():
    print(f"\n{'='*60}\n{nom}\n{'='*60}")

    dossier_source = os.path.join(DOSSIER_SOURCES, nom)

    # Charger les pages déjà téléchargées si présentes, sinon télécharger
    if os.path.exists(dossier_source) and len(os.listdir(dossier_source)) > 0:
        pages = sorted([
            os.path.join(dossier_source, f)
            for f in os.listdir(dossier_source) if f.endswith(".jpg")
        ])
        print(f"{len(pages)} pages déjà disponibles")
    else:
        pages = telecharger_pages_iiif(
            f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest",
            dossier_source,
            prefixe=nom
        )

    # Segmenter les illustrations
    segmenter_corpus(
        pages,
        os.path.join(DOSSIER_SEG, nom),
        modele_yolo,
        conf_thres=0.25
    )

## 6. Libérer le modèle YOLO

In [ ]:
liberer_yolo(modele_yolo)
print("YOLO libéré")

## 7. Récapitulatif

In [ ]:
print("Récapitulatif :\n")
for bsb_id, nom in SOURCES_BSB.items():
    chemin = os.path.join(DOSSIER_SEG, nom)
    if os.path.exists(chemin):
        n = len([f for f in os.listdir(chemin) if f.endswith(".jpg")])
        print(f"  {nom:45s} : {n} illustrations")
    else:
        print(f"  {nom:45s} : (non traité)")

# Partie B — Constitution du corpus des Bibles illustrées (MDZ, 1551-1750)

Cette partie interroge le portail de recherche MDZ pour identifier tous les ouvrages correspondant à un thème (ici les Bibles illustrées entre 1551 et 1750), permet de sélectionner manuellement les documents pertinents dans un tableau interactif, puis extrait automatiquement un nombre donné d'illustrations par ouvrage retenu.

## 8. Recherche des documents sur le portail MDZ

Interroge l'API de recherche (`digitale-sammlungen.de/api/search`) avec la requête `(biblia tafeln)` et des filtres sur la date (1551-1750) et le type de fabrication (hors manuscrits), en paginant explicitement page par page via `startPage` — l'API plafonne à 250 résultats par requête quel que soit le `pageSize` demandé, cette pagination est donc nécessaire pour récupérer l'intégralité des résultats. Le résultat complet est stocké dans `docs_final`.

Pour adapter la recherche à un autre thème : modifier `query` et `filtres`.

In [2]:
import requests, re

def recuperer_pages(query, filtres=None, page_size=100, page_debut=1, page_fin=20):
    """Récupère les documents des pages page_debut à page_fin (incluses).
    Affiche le détail de chaque page sur une ligne distincte."""
    url_api = "https://www.digitale-sammlungen.de/api/search"
    tous = []

    for sp in range(page_debut, page_fin + 1):
        params = {
            "query": query, "handler": "simple-metadata", "ocrContext": 1,
            "sortField": "date", "sortOrder": "asc",
            "startPage": sp, "pageSize": page_size,
        }
        if filtres:
            params["filter"] = filtres

        r = requests.get(url_api, params=params, timeout=30)
        if r.status_code != 200:
            print(f"  startPage {sp:2d} → HTTP {r.status_code}")
            continue

        docs = r.json().get("docs", [])
        for d in docs:
            tous.append({
                "id"    : d.get("id"),
                "titre" : re.sub(r"<[^>]+>", "", d.get("title", "")),
                "lieu"  : ", ".join(d.get("publicationPlaces", [])),
                "date"  : d.get("publicationDate", ""),
            })
        # Une ligne par page : nb de docs de la page + cumul
        print(f"  startPage {sp:2d} → {len(docs):3d} docs sur cette page | cumul : {len(tous)}")

    print(f"\n✓ {len(tous)} documents récupérés (pages {page_debut} à {page_fin})")
    return tous


docs_final = recuperer_pages(
    query="(biblia tafeln)",
    filtres=['type_manufact:-"handmade"', 'date_facet:[1551-01-01 TO 1750-06-16]'],
    page_debut=0, page_fin=20,
)

  startPage  0 → 100 docs sur cette page | cumul : 100


  startPage  1 → 100 docs sur cette page | cumul : 200


  startPage  2 → 100 docs sur cette page | cumul : 300


  startPage  3 → 100 docs sur cette page | cumul : 400


  startPage  4 → 100 docs sur cette page | cumul : 500


...

  startPage 16 → 100 docs sur cette page | cumul : 1700


  startPage 17 → 100 docs sur cette page | cumul : 1800


  startPage 18 → 100 docs sur cette page | cumul : 1900


  startPage 19 →  45 docs sur cette page | cumul : 1945


  startPage 20 →   0 docs sur cette page | cumul : 1945

✓ 1945 documents récupérés (pages 0 à 20)


## 9. Tableau de sélection interactif (HTML)

Génère un fichier HTML autonome (`resultats/selection_bibles_mdz.html`) listant tous les documents trouvés à l'étape précédente. Ouvrir ce fichier dans un navigateur permet de retirer les documents non pertinents (croix rouge, sauvegarde automatique dans le navigateur), puis de télécharger la sélection finale sous forme de fichier texte — à conserver pour l'étape suivante.

## 10. Extraction en masse des illustrations

Lit la liste d'identifiants BSB retenue (fichier texte issu de l'étape précédente ; adapter le chemin `chemin_selection` à l'emplacement réel du fichier), puis pour chaque document : télécharge les pages via IIIF et segmente les illustrations avec YOLO, jusqu'à `n_illustrations` par ouvrage. Gère la reprise (documents déjà traités automatiquement ignorés) et les erreurs 429 (rate-limit) avec un retry progressif. Les illustrations sont sauvegardées dans `data/bibles_mdz/segmentees/{bsb_id}/`.

## 11. Statistiques finales du corpus

Parcourt le dossier de sortie et calcule le bilan global : nombre de Bibles traitées, avec/sans illustration détectée, total d'illustrations extraites et moyenne par ouvrage.

In [3]:
# ── Tableau HTML de sélection — 1944 Bibles, suppression + sauvegarde + export ──
import os, json

chemin_html = os.path.join(RACINE, "resultats", "Bibles_mdz", "selection_bibles_mdz.html")

docs_json = json.dumps([
    {"id": d["id"], "date": d["date"] or "", "lieu": d["lieu"] or "", "titre": d["titre"] or ""}
    for d in docs_final
], ensure_ascii=False)

html = f"""<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<title>Sélection des Bibles illustrées — MDZ</title>
<style>
  body {{ font-family: Georgia, serif; margin: 30px; color: #2c2c2c; background: #faf8f5; }}
  h1 {{ font-size: 22px; color: #5a3e2b; border-bottom: 2px solid #c89b6a; padding-bottom: 10px; }}
  .sous-titre {{ color: #777; font-size: 14px; margin-bottom: 20px; }}
  .barre {{ position: sticky; top: 0; background: #faf8f5; padding: 15px 0; z-index: 10;
            border-bottom: 1px solid #ddd; margin-bottom: 10px; }}
  button {{ background: #5a3e2b; color: white; border: none; padding: 10px 20px; font-size: 14px;
            border-radius: 4px; cursor: pointer; font-family: Georgia, serif; margin-right: 10px; }}
  button:hover {{ background: #7a5638; }}
  .reset {{ background: #999; }}
  #compteur {{ margin-left: 5px; color: #777; font-size: 13px; }}
  table {{ border-collapse: collapse; width: 100%; background: white; }}
  th {{ background: #5a3e2b; color: white; padding: 10px; font-size: 13px; text-align: left;
        position: sticky; top: 70px; }}
  td {{ padding: 8px 10px; border-bottom: 1px solid #eee; font-size: 13px; vertical-align: middle; }}
  tr:hover {{ background: #f5efe6; }}
  .num {{ color: #aaa; font-size: 12px; width: 45px; }}
  .ident {{ font-family: monospace; font-size: 12px; color: #555; }}
  .titre {{ max-width: 430px; }}
  .suppr {{ background: #c0392b; color: white; border: none; border-radius: 50%;
            width: 26px; height: 26px; cursor: pointer; font-size: 14px; line-height: 1; }}
  .suppr:hover {{ background: #e74c3c; }}
</style>
</head>
<body>
  <h1>Sélection des Bibles illustrées (MDZ)</h1>
  <p class="sous-titre">
    Cliquez sur la croix rouge pour retirer les documents qui ne vous intéressent pas.
    Votre sélection est <b>sauvegardée automatiquement</b> — vous pouvez fermer et reprendre.
    Quand vous avez terminé, cliquez sur « Télécharger ma sélection » et envoyez-moi le fichier.
  </p>
  <div class="barre">
    <button onclick="telecharger()">⬇ Télécharger ma sélection</button>
    <button class="reset" onclick="reinitialiser()">Tout réafficher</button>
    <span id="compteur"></span>
  </div>
  <table>
    <thead>
      <tr><th class="num">#</th><th>Identifiant</th><th>Date</th><th>Lieu</th><th>Titre</th><th></th></tr>
    </thead>
    <tbody id="corps"></tbody>
  </table>

<script>
  const DOCS = {docs_json};
  const STORAGE_KEY = 'selection_bibles_mdz';

  function charger() {{
    const d = localStorage.getItem(STORAGE_KEY);
    return d ? JSON.parse(d) : [];
  }}
  function sauver(supprimes) {{
    localStorage.setItem(STORAGE_KEY, JSON.stringify(supprimes));
  }}

  function rendre() {{
    const supprimes = charger();
    const corps = document.getElementById('corps');
    corps.innerHTML = '';
    let n = 0;
    DOCS.forEach(doc => {{
      if (supprimes.includes(doc.id)) return;
      n++;
      const tr = document.createElement('tr');
      tr.innerHTML =
        '<td class="num">' + n + '</td>' +
        '<td class="ident">' + doc.id + '</td>' +
        '<td>' + doc.date + '</td>' +
        '<td>' + doc.lieu + '</td>' +
        '<td class="titre">' + doc.titre + '</td>' +
        '<td><button class="suppr" title="Retirer">✕</button></td>';
      tr.querySelector('.suppr').addEventListener('click', () => {{
        const s = charger();
        s.push(doc.id);
        sauver(s);
        rendre();
      }});
      corps.appendChild(tr);
    }});
    document.getElementById('compteur').textContent =
      n + ' document(s) conservé(s) sur ' + DOCS.length;
  }}

  function telecharger() {{
    const supprimes = charger();
    const gardes = DOCS.filter(d => !supprimes.includes(d.id));
    let txt = 'SÉLECTION DES BIBLES ILLUSTRÉES — MDZ\\n';
    txt += gardes.length + ' documents conservés sur ' + DOCS.length + '\\n';
    txt += '='.repeat(70) + '\\n\\n';
    gardes.forEach((d, i) => {{
      txt += (i+1) + '. ' + d.id + ' | ' + d.date + ' | ' + d.lieu + ' | ' + d.titre + '\\n';
    }});
    const blob = new Blob([txt], {{ type: 'text/plain;charset=utf-8;' }});
    const lien = document.createElement('a');
    lien.href = URL.createObjectURL(blob);
    lien.download = 'selection_bibles_celine.txt';
    lien.click();
  }}

  function reinitialiser() {{
    if (confirm('Réafficher tous les documents (annuler vos suppressions) ?')) {{
      localStorage.removeItem(STORAGE_KEY);
      rendre();
    }}
  }}

  rendre();
</script>
</body>
</html>"""

with open(chemin_html, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✓ Tableau de sélection généré : {chemin_html}")
print(f"  {len(docs_final)} documents")

✓ Tableau de sélection généré : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/Bibles_mdz/selection_bibles_mdz.html
  1945 documents


In [ ]:
import os, sys, re, requests, time
from PIL import Image
from io import BytesIO

RACINE = os.path.abspath("../../")
sys.path.insert(0, os.path.join(RACINE, "notebooks"))
sys.path.insert(0, RACINE)
sys.path.insert(0, os.path.join(RACINE, "yolov5_repo"))

from gallica_utils import segmenter_page, charger_yolo, liberer_yolo
print("✓ Imports OK")

DOSSIER_BIBLES_SEG = os.path.join(RACINE, "data", "bibles_mdz", "segmentees")
os.makedirs(DOSSIER_BIBLES_SEG, exist_ok=True)

# ── Reprise : nb d'illustrations déjà extraites ─────────────
def deja_traite(bsb_id):
    dossier_seg = os.path.join(DOSSIER_BIBLES_SEG, bsb_id)
    if not os.path.isdir(dossier_seg):
        return None
    illus = [f for f in os.listdir(dossier_seg)
             if f.lower().endswith((".jpg", ".jpeg", ".png"))
             and not f.startswith("_tmp_")]
    return len(illus)

# ── Téléchargement avec gestion du rate-limit 429 ───────────
def telecharger_avec_retry(url, max_essais=4, pause_base=2):
    for essai in range(max_essais):
        try:
            response = requests.get(url, timeout=20)
            if response.status_code == 429:
                attente = pause_base * (2 ** essai)
                print(f"      429 — pause {attente}s", end="\r")
                time.sleep(attente)
                continue
            response.raise_for_status()
            return Image.open(BytesIO(response.content)).convert("RGB")
        except requests.exceptions.HTTPError:
            raise
        except Exception:
            time.sleep(pause_base)
    raise Exception("Échec après plusieurs essais (rate-limit persistant)")

# ── Extraction de N illustrations pour un document ──────────
def recuperer_n_illustrations(bsb_id, modele_yolo, n_illustrations=10, conf_thres=0.25):
    url = f"https://api.digitale-sammlungen.de/iiif/presentation/v2/{bsb_id}/manifest"
    try:
        manifest = requests.get(url, timeout=20).json()
    except Exception as e:
        print(f"  ✗ Manifest inaccessible : {e}")
        return 0
    canvases = manifest["sequences"][0]["canvases"]
    dossier_seg = os.path.join(DOSSIER_BIBLES_SEG, bsb_id)
    os.makedirs(dossier_seg, exist_ok=True)
    total_illus = 0
    for i, canvas in enumerate(canvases):
        if total_illus >= n_illustrations:
            break
        page_num = i + 1
        img_url  = canvas["images"][0]["resource"]["@id"]
        try:
            img = telecharger_avec_retry(img_url)
        except Exception as e:
            print(f"    page {page_num} ignorée : {e}")
            continue
        chemin_tmp = os.path.join(dossier_seg, f"_tmp_{bsb_id}_p{page_num:03d}.jpg")
        try:
            img.save(chemin_tmp)
            prefixe = f"{bsb_id}_page{page_num:03d}"
            nb = segmenter_page(chemin_tmp, prefixe, dossier_seg, modele_yolo, conf_thres)
            total_illus += nb
        except Exception as e:
            print(f"    erreur segmentation page {page_num} : {e}")
        finally:
            if os.path.exists(chemin_tmp):
                os.remove(chemin_tmp)
        time.sleep(0.5)
    return total_illus

# ── Lire la sélection ───────────────────────────────────────
chemin_selection = os.path.join(RACINE, "retours_celine", "selection_bibles_celine_1550-1750_Biblia.txt")
with open(chemin_selection, encoding="utf-8") as f:
    contenu = f.read()
ids = re.findall(r"bsb\d+", contenu)
print(f"{len(ids)} documents dans la sélection\n")

# ── Pipeline avec reprise ───────────────────────────────────
modele_yolo = charger_yolo()
resultats   = {}
nb_sautes   = 0

for k, bsb_id in enumerate(ids, 1):
    deja = deja_traite(bsb_id)
    if deja is not None:
        resultats[bsb_id] = deja
        nb_sautes += 1
        print(f"[{k}/{len(ids)}] {bsb_id} → déjà traité ({deja} illus), ignoré")
        continue
    print(f"[{k}/{len(ids)}] {bsb_id}", end=" → ")
    try:
        n = recuperer_n_illustrations(bsb_id, modele_yolo, n_illustrations=10)
        resultats[bsb_id] = n
        print(f"{n} illustrations")
    except Exception as e:
        resultats[bsb_id] = -1
        print(f"ERREUR : {e}")

liberer_yolo(modele_yolo)

# ── Récapitulatif ───────────────────────────────────────────
ok          = sum(1 for v in resultats.values() if v > 0)
vides       = sum(1 for v in resultats.values() if v == 0)
err         = sum(1 for v in resultats.values() if v == -1)
total_illus = sum(v for v in resultats.values() if v > 0)
print(f"\n{'='*55}")
print(f"✓ Terminé — {ok} docs avec illustrations, {vides} sans, {err} en erreur")
print(f"  {nb_sautes} documents déjà traités (sautés)")
print(f"  Total illustrations extraites : {total_illus}")

In [2]:
import os

DOSSIER_BIBLES_SEG = os.path.join(RACINE, "data", "bibles_mdz", "segmentees")

total_illus  = 0
total_bibles = 0
avec_illus   = 0
sans_illus   = 0

for bsb_id in sorted(os.listdir(DOSSIER_BIBLES_SEG)):
    dossier = os.path.join(DOSSIER_BIBLES_SEG, bsb_id)
    if not os.path.isdir(dossier):
        continue
    illus = [f for f in os.listdir(dossier)
             if f.lower().endswith((".jpg", ".jpeg", ".png"))
             and not f.startswith("_tmp_")]
    n = len(illus)
    total_bibles += 1
    total_illus  += n
    if n > 0:
        avec_illus += 1
    else:
        sans_illus += 1

print("="*50)
print(f"  Bibles traitées      : {total_bibles}")
print(f"    avec illustrations : {avec_illus}")
print(f"    sans illustration  : {sans_illus}")
print(f"  TOTAL illustrations  : {total_illus}")
print("="*50)
if avec_illus:
    print(f"  Moyenne par Bible (avec illus) : {total_illus/avec_illus:.1f}")

  Bibles traitées      : 398
    avec illustrations : 390
    sans illustration  : 8
  TOTAL illustrations  : 3570
  Moyenne par Bible (avec illus) : 9.2
